# Antibody-drug conjugate: IgG1 Fc (6P6D)

Covalently attach a drug-linker to **ASN297 of both heavy chains** of an IgG1 Fc
(PDB 6P6D) and build one GROMACS topology for the conjugate.

`PolyLego-BuildConjugate` runs `pdb2gmx` on the protein, parametrises the modifier with
GAFF2 (AmberTools), derives bonded terms for each covalent junction from a parametrised
junction fragment, merges everything, and relaxes the conjugate with a vacuum EM.

Runtime: about 2 minutes.

## Setup
Checks that PolyLego, GROMACS and AmberTools are available and pins the thread counts.

In [1]:
import os, sys, json
sys.path.insert(0, os.path.abspath(os.path.join("..", "common")))
from polylego_examples import *

# Pin threads for every step launched from this notebook (see examples/README.md).
os.environ.setdefault("OMP_NUM_THREADS", "8")
for k in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[k] = "1"
os.environ.setdefault("PYTHONHASHSEED", "0")

GMX = check_env()
os.environ["POLYLEGO_GMX"] = GMX
# GPU for the GROMACS runs: "auto" (use a compatible GPU if there is one, else CPU),
# "none" (CPU only), or a GPU id such as "0".
GPU = os.environ.get("POLYLEGO_GPU", "auto")
os.makedirs("output", exist_ok=True)

PolyLego   1.4.0  ($CONDA_PREFIX/lib/python3.10/site-packages/PolyLego)
Python     3.10.13
GROMACS    $CONDA_PREFIX/bin/gmx
AmberTools $CONDA_PREFIX/bin/antechamber
OMP_NUM_THREADS        8
OPENBLAS_NUM_THREADS   1
MKL_NUM_THREADS        1


## 1. Inputs
The drug-linker. The fluorine is a placeholder marking the attachment point; it is removed when the bond is formed.

In [2]:
show_mol("input/ligand.sdf")

[3D view - run the notebook in Jupyter to display it]


Two conjugation sites: same residue number on two chains, told apart by `attach_protein_occurrence`.

In [3]:
print(open("input/conjugation_spec.json").read())

[
  {
    "modifier_file": "ligand.sdf",
    "mol_name": "LIG1",
    "attach_protein_resnum": 297,
    "attach_protein_atomname": "ND2",
    "attach_protein_occurrence": 1,
    "attach_modifier_idx": 0,
    "deletor_modifier_idx": 99,
    "deletor_protein_atomname": "HD21"
  },
  {
    "modifier_file": "ligand.sdf",
    "mol_name": "LIG2",
    "attach_protein_resnum": 297,
    "attach_protein_atomname": "ND2",
    "attach_protein_occurrence": 2,
    "attach_modifier_idx": 0,
    "deletor_modifier_idx": 99,
    "deletor_protein_atomname": "HD21"
  }
]



In [4]:
show_complex("input/6p6d.pdb", ligand_resn=("ASN",), focus_ligand=False)

[3D view - run the notebook in Jupyter to display it]


## 2. Prepare the protein
`pdb2gmx` only types standard residues, so the glycans and other HETATM records of the crystal structure are removed.

In [5]:
PROTEIN = strip_hetatm("input/6p6d.pdb", "output/6p6d_protein.pdb")

## 3. Build the conjugate

In [6]:
run(["PolyLego-BuildConjugate", "--protein", PROTEIN,
     "--conjugations", "input/conjugation_spec.json", "--output_path", "output",
     "--gmx", GMX, "--gpu", GPU],
    log="output/conjugate.log")

$ PolyLego-BuildConjugate --protein output/6p6d_protein.pdb --conjugations input/conjugation_spec.json --output_path output --gmx $CONDA_PREFIX/bin/gmx --gpu none


  PolyLego 1.4.0 | BuildConjugate
  2 conjugation site(s) on 6p6d_protein.pdb
  protein topology (pdb2gmx)
  modifier parameters and cross-links
  vacuum energy minimisation
  closest heavy-atom distance after EM: 2.53 A
  Finished in 1 min 55 s
    output/conjugate_top/conjugate.top
    output/em/em/em.gro
  -> exit 0, 117 s


## 4. Result
Conjugate after vacuum EM (modifiers in green):

In [7]:
TOP, GRO = "output/conjugate_top/conjugate.top", "output/em/em/em.gro"
show_gro(TOP, GRO, "output", ligand_resn=("LIG1", "LIG2"))

[3D view - run the notebook in Jupyter to display it]


Protein-modifier bonds: length after the vacuum EM, and the equilibrium length `b0` assigned to that bond in the topology.

In [8]:
import numpy as np, parmed
def crosslinks(top, gro, modifier_resn):
    """Covalent bonds between the protein and each modifier, with their length after EM."""
    s = parmed.load_file(top, xyz=gro)
    rows = []
    for b in s.bonds:
        a1, a2 = b.atom1, b.atom2
        r1, r2 = a1.residue.name, a2.residue.name
        if (r1 in modifier_resn) != (r2 in modifier_resn):
            p, m = (a2, a1) if r1 in modifier_resn else (a1, a2)
            d = np.linalg.norm(np.array([p.xx, p.xy, p.xz]) - np.array([m.xx, m.xy, m.xz]))
            b0 = b.type.req if b.type is not None else float("nan")
            rows.append((f"{p.residue.name}{p.residue.number}:{p.name}",
                         f"{m.residue.name}:{m.name}", round(float(d), 3), round(float(b0), 3)))
    print(f"  {'protein atom':14s}    {'modifier':10s}  {'length':>7s}  {'b0':>6s}")
    for r in rows:
        print(f"  {r[0]:14s} -- {r[1]:10s}  {r[2]:6.3f} A  {r[3]:5.3f} A")
    return rows

_ = crosslinks(TOP, GRO, {"LIG1", "LIG2"})

  protein atom      modifier     length      b0
  ASN60:ND2      -- LIG1:C1      1.527 A  1.522 A
  ASN269:ND2     -- LIG2:C1      1.589 A  1.522 A
